In [1]:
import pandas as pd

In [2]:
# Step 1: aggregate per site_id, archetype, material_category
# site_sum = (
#     df.groupby(["site_id", "archetypes", "material_category"], as_index=False)
#       .agg(site_intensity=("kg_t_ore", "sum"))
# )
#
# # Step 2: compute min / mean / max across sites for each archetype + material_category
# stats = (
#     site_sum.groupby(["archetypes", "material_category"])
#             .agg(
#                 n_sites=("site_id", "nunique"),
#                 min_kg_per_kg=("site_intensity", "min"),
#                 mean_kg_per_kg=("site_intensity", "mean"),
#                 max_kg_per_kg=("site_intensity", "max"),
#             )
#             .reset_index()
#)

In [3]:
# Cleaned data
energy_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\energy_df.xlsx')
material_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\material_df.xlsx')
biosphere_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\biosphere_df.xlsx')
land_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\land_df.xlsx')
carbon_stock_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\carbon_stock_df.xlsx')

In [4]:
# Prices and production data
price_df = pd.read_excel(r'data/Prices/Prices_data.xlsx', sheet_name='data')
production_df = pd.read_excel(r'data/MetalliCan/sites_for_lci.xlsx', sheet_name='prod_data')

In [5]:
from utils.data_manipulations import build_activity_name, add_site_id

In [6]:
# Add site_id to dataframes
production_df = add_site_id(production_df)
energy_df = add_site_id(energy_df)
material_df = add_site_id(material_df)
biosphere_df = add_site_id(biosphere_df)
land_df = add_site_id(land_df)
carbon_stock_df = add_site_id(carbon_stock_df)

In [7]:
energy_id_to_remove = [
# less than 50% of NRJ = GHG
'BC-MAIN-599152a0', # Copper Mountain (cu concentrate)
'ON-MAIN-1f126a43', # Macassa (doré)
'ON-MAIN-7f050560', # Red Lake (doré)
'GRP-147b3123', # Timmins Operation (doré)
'ON-MAIN-7607a50e', # Young Davidson (doré)

# more than 1;5 x
'NU-MAIN-8b0264c9', # Meliadine (doré)
'QC-MAIN-9de9bb0d', # Kiena (doré)
'ON-MAIN-cb85213a', # Eagle River (doré)
'ON-MAIN-6e9be24e' # Hemlo (Williams)
]

for GRP-14bfbb82 change Other to Diesel

In [8]:
# Remove the energy flows for these ones
energy_df = energy_df[~energy_df['site_id'].isin(energy_id_to_remove)]

In [9]:
# Add activitiy_name to production_df
production_df['activity_name'] = production_df.apply(lambda row: build_activity_name(row, production_df), axis=1)

In [10]:
energy_df = energy_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')
material_df = material_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')
biosphere_df = biosphere_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')
land_df = land_df.merge(production_df[['site_id', 'activity_name','archetypes']], on='site_id', how='left')
carbon_stock_df = carbon_stock_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')

In [11]:
# To avoid double counting
biosphere_df = biosphere_df[~((biosphere_df['source_id'] == 'https://www.canada.ca/en/environment-climate-change/services/environmental-indicators/greenhouse-gas-emissions/large-facilities.html; https://open.canada.ca/data/en/dataset/a8ba14b7-7f23-462a-bdbb-83b0ef629823') & (biosphere_df['substance_id'] == 'NA - GHG'))]

In [12]:
# Only keep water consumption
biosphere_df = biosphere_df[~biosphere_df['flow_direction'].isin(['Withdrawal', 'Discharged'])]

In [13]:
# Assign 'substance_name' to 'substance_id' for CO2, CH4 and N2O
# substance_dict = {
#     '124-38-9': 'Carbon Dioxide',
#     '74-82-8': 'Methane',
#     '10024-97-2': 'Nitrous Oxide',
# }
#
# mask = biosphere_df['substance_id'].isin(substance_dict)
# biosphere_df.loc[mask, 'substance_name'] = (
#     biosphere_df.loc[mask, 'substance_id'].map(substance_dict)
# )

In [14]:
substance_CO2 = ['124-38-9']
release_pathway = ['Stationary Fuel Combustion', 'On-site Transportation']
CO2_df = biosphere_df[
    biosphere_df['substance_id'].isin(substance_CO2) &
    biosphere_df['release_pathway'].isin(release_pathway)
]

# Data cleaning and integration

In [15]:
energy_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'flow_type', 'subflow_type', 'value_MJ']
material_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'flow_type', 'subflow_type', 'mass_t']
biosphere_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'compartment_name', 'substance_name', 'flow_direction', 'release_pathway', 'unit', 'value']
land_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'area_m2', 'operation_periods']

In [16]:
energy_df = energy_df[energy_col]
material_df = material_df[material_col]
biosphere_df = biosphere_df[biosphere_col]
CO2_df = CO2_df[biosphere_col]
land_df = land_df[land_col]

## Create a tailings df

In [17]:
# We create a tailings_df to add the quantity of tailings as a negative material flow
tailings_df = production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'Stream', 'tailings_t_mass']]
tailings_df['flow_type'] = 'Material use'
tailings_df['subflow_type'] = 'Tailings'
tailings_df['unit'] = 't'
tailings_df['value'] = - tailings_df['tailings_t_mass']  # negative value for output in LCI
tailings_df.drop(columns=['tailings_t_mass'], inplace=True)

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_3832\54366009.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tailings_df['flow_type'] = 'Material use'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_3832\54366009.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tailings_df['subflow_type'] = 'Tailings'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_3832\54366009.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value inste

In [18]:
def get_tailings_subflow_type(row):
    stream = row['Stream']

    if not isinstance(stream, str):
        return 'Tailings|Other'

    if 'Au-Ag dore' in stream:
        return 'Tailings|Gold'
    elif stream == 'Ni-Cu bulk concentrates':
        return 'Tailings|Nickel'
    elif 'Cu concentrates' or 'Cu and Mo concentrates' or 'Cu and Zn concentrates with Ag credits' in stream:
        return 'Tailings|Copper'
    elif 'Yellowcake' in stream:
        return 'Tailings|Uranium'
    else:
        return 'Tailings|Other'

In [19]:
tailings_df['subflow_type'] = tailings_df.apply(get_tailings_subflow_type, axis=1)

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_3832\4113187377.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tailings_df['subflow_type'] = tailings_df.apply(get_tailings_subflow_type, axis=1)


## Cleaning

In [20]:
# Maybe need to differentiate unit and value in energy_df and material_df ?
energy_df['unit'] = 'MJ'
material_df['unit'] = 't'
#tailings_df['unit'] = 't'
land_df['unit'] = 'm2'
energy_df.rename(columns={'value_MJ': 'value'}, inplace=True)
material_df.rename(columns={'mass_t': 'value'}, inplace=True)
land_df.rename(columns={'area_m2': 'value'}, inplace=True)

In [21]:
# To know the % inferred vs 'primary'
energy_df['data_source'] = 'MetalliCan'
material_df['data_source'] = 'MetalliCan'
biosphere_df['data_source'] = 'MetalliCan'
CO2_df['data_source'] = 'MetalliCan'
land_df['data_source'] = 'MetalliCan'
carbon_stock_df['data_source'] = 'MetalliCan'

In [22]:
# To know the % inferred vs 'primary'
energy_df['value_formula'] = 'MetalliCan'
material_df['value_formula'] = 'MetalliCan'
biosphere_df['value_formula'] = 'MetalliCan'
CO2_df['value_formula'] = 'MetalliCan'
land_df['value_formula'] = 'MetalliCan'
carbon_stock_df['value_formula'] = 'MetalliCan'

# Data-gap filling

In [23]:
from core.data_gap_filling import *

In [24]:
# Initialize the InferenceEngine Class
engine = InferenceEngine(
    site_df=production_df,
    production_df=production_df,
    energy_df=energy_df,
    co2_df=CO2_df,
    land_df=land_df,
    material_df=material_df,
)

## Energy

In [25]:
site_id_to_fill_nrj = production_df[production_df['infer_energy_data'] == 'Yes']['site_id'].tolist()

In [26]:
ef = {
    "diesel": 2681,
    "natural_gas": 2354,
    "lpg": 2753
}

stationary_share_rules = {
    "Open-pit, concentrator": {"diesel": 0.7, "natural_gas": 0.2, "lpg": 0.1},
    "Underground, concentrator": {"diesel": 0.7, "natural_gas": 0.2, "lpg": 0.1},
}

default_shares = {"diesel": 0.3, "natural_gas": 0.6, "lpg": 0.1}

In [27]:
combined_energy_df, inferred_energy_df = engine.infer_energy_for_sites(
    site_ids=site_id_to_fill_nrj,
    ef_co2_per_unit=ef,
    stationary_share_rules=stationary_share_rules,
    default_shares=default_shares
)


In [28]:
inferred_energy_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula
0,QC-MAIN-089f3c60,Open-pit mining at Bloom Lake,Open-pit,Magnetite concentrator,Energy,Diesel|Transport,3.718453e+07,L,Inference from CO2 (transport diesel),(99691.72 * 1e6) / 2681
1,QC-MAIN-089f3c60,Open-pit mining at Bloom Lake,Open-pit,Magnetite concentrator,Energy,Diesel|Stationary,1.418597e+06,L,Inference from CO2 (stationary diesel),(3803.259 * 1e6) / 2681
2,QC-MAIN-089f3c60,Open-pit mining at Bloom Lake,Open-pit,Magnetite concentrator,Energy,Natural Gas|Stationary,3.231316e+06,m3,Inference from CO2 (stationary natural_gas),(7606.518 * 1e6) / 2354
3,QC-MAIN-089f3c60,Open-pit mining at Bloom Lake,Open-pit,Magnetite concentrator,Energy,Lpg|Stationary,4.604987e+05,L,Inference from CO2 (stationary lpg),(1267.7530000000002 * 1e6) / 2753
4,BC-MAIN-bf503b6b,Open-pit mining and beneficiation at Highland ...,"Open-pit, concentrator",Cu polymetallic,Energy,Diesel|Transport,6.003123e+07,L,Inference from CO2 (transport diesel),(160943.741 * 1e6) / 2681
5,BC-MAIN-bf503b6b,Open-pit mining and beneficiation at Highland ...,"Open-pit, concentrator",Cu polymetallic,Energy,Diesel|Stationary,4.329585e+06,L,Inference from CO2 (stationary diesel),(11607.616369999998 * 1e6) / 2681
6,BC-MAIN-bf503b6b,Open-pit mining and beneficiation at Highland ...,"Open-pit, concentrator",Cu polymetallic,Energy,Natural Gas|Stationary,1.408862e+06,m3,Inference from CO2 (stationary natural_gas),(3316.46182 * 1e6) / 2354
7,BC-MAIN-bf503b6b,Open-pit mining and beneficiation at Highland ...,"Open-pit, concentrator",Cu polymetallic,Energy,Lpg|Stationary,6.023360e+05,L,Inference from CO2 (stationary lpg),(1658.23091 * 1e6) / 2753
8,NL-MAIN-d9036091,Refining at Long Harbour,Refinery,Ni hydrometallurgical refinery,Energy,Diesel|Transport,9.977881e+05,L,Inference from CO2 (transport diesel),(2675.07 * 1e6) / 2681
9,NL-MAIN-d9036091,Refining at Long Harbour,Refinery,Ni hydrometallurgical refinery,Energy,Diesel|Stationary,2.241697e+06,L,Inference from CO2 (stationary diesel),(6009.99 * 1e6) / 2681


## Material

In [29]:
site_id_to_fill_material = production_df[production_df['infer_material_data'] == 'Yes']['site_id'].tolist()

In [30]:
material_archetype_rules_df = pd.read_excel(r'data/SI/SI_data_gap_filling.xlsx', sheet_name='DATA')

In [31]:
material_archetype_rules_df

,archetype,flow_type,material_name,ei_activity,role_process_stage,value,unit,reference_flow,distribution,source
0,Au–Ag free-milling,Materials,Activated carbon,"activated carbon, granular",Adsorption of Au from solution (CIP/CIL),0.050000,kg,ore_processed_t,NaN,911Metallurgist
1,Au–Ag polymetallic,Materials,Activated carbon,"activated carbon, granular",Gold adsorption,0.050000,kg,ore_processed_t,NaN,911Metallurgist
2,Au–Ag refractory,Materials,Activated carbon,"activated carbon, granular",Gold adsorption in CIP/CIL,0.050000,kg,ore_processed_t,NaN,911Metallurgist
3,High-grade U mine + mill,Materials,Ammonia,"market for ammonia, anhydrous, liquid",Yellowcake precipitation (ADU),6.000000,kg,ore_processed_t,NaN,Uranium metallurgy handbooks
4,Ni–Cu sulfide,Materials,Carboxymethyl cellulose,"market for carboxymethyl cellulose, powder",Talc/serpentine depression,0.200000,kg,ore_processed_t,NaN,Raglan concentrator
5,Au–Ag polymetallic,Materials,Copper sulfate,market for copper sulfate,Zn activation,0.100000,kg,ore_processed_t,NaN,Cu–Zn flotation literature
6,Cu polymetallic,Materials,Copper sulfate,market for copper sulfate,Zn activation,0.100000,kg,ore_processed_t,NaN,Cu–Zn flotation
7,Ni–Cu sulfide,Materials,Copper sulfate,market for copper sulfate,Pentlandite activation,0.050000,kg,ore_processed_t,NaN,Nickel flotation literature
8,Cu Porphyry,Materials,Flocculant,market for polyacrylamide,Tailings thickening,0.005000,kg,ore_processed_t,NaN,Thickener design manuals
9,Au–Ag polymetallic,Materials,Frother,"market for chemical, organic",Flotation,0.020000,kg,ore_processed_t,NaN,Wills & Finch


In [32]:
engine.init_material_inference(material_archetype_rules_df)

In [33]:
combined_material_df, inferred_material_df = engine.infer_material_for_sites(
    site_ids=site_id_to_fill_material,
    overwrite=False,
)

⚠️ No material rules for archetype 'Magnetite concentrator'
⚠️ No ore_processed_t for site QC-MAIN-de3d8b7b
⚠️ No material rules for archetype 'Zn refinery'
⚠️ No material rules for archetype 'Fe concentrator + pellet plant'
⚠️ No ore_processed_t for site ON-MAIN-63b394c3
⚠️ No material rules for archetype 'Cu-Ni smelter and refinery'
⚠️ No ore_processed_t for site QC-MAIN-30c1828c
⚠️ No material rules for archetype 'Cu smelter'
⚠️ No ore_processed_t for site NL-MAIN-d9036091
⚠️ No material rules for archetype 'Ni hydrometallurgical refinery'
⚠️ No material rules for archetype 'Direct Shipping Ore'
⚠️ No material rules for archetype 'Nb mine'
⚠️ No ore_processed_t for site QC-MAIN-649d2873
⚠️ No material rules for archetype 'Nb smelter'
⚠️ No material rules for archetype 'Fe concentrator'
⚠️ No ore_processed_t for site ON-MAIN-40ce0593
⚠️ No material rules for archetype 'Ni-Cu smelter'
⚠️ No ore_processed_t for site AB-MAIN-d3a4aba9
⚠️ No material rules for archetype 'Co hydrometallurg

In [35]:
combined_material_df['flow_type'].unique()

array(['Material use', 'Materials', 'Water'], dtype=object)

## Cement

In [38]:
site_id_to_fill_material = production_df[production_df['infer_material_data'] == 'Yes']['site_id'].tolist()

In [39]:
cement_params = {
    "underground": {
        "cement_factor": (5, 50),
        "backfill_share": (0.3, 0.9),  # optional, future
    },
    "open_pit": None,
}


In [40]:
combined_cement_df, inferred_cement_df = engine.infer_cement_for_sites(
    site_ids=site_id_to_fill_material,
    cement_params=cement_params,
)

## Explosives

In [46]:
site_id_to_fill_explosives = production_df[production_df['infer_material_data'] == 'Yes']['site_id'].tolist()

In [47]:
explosives_params = {
    "open_pit": {
        "strip_ratio": (1.5, 6.0),       # t waste / t ore
        "explosive_factor": (0.25, 0.8), # kg explosives / t material
    },
    "underground": {
        "explosive_factor": (0.1, 0.3),  # kg explosives / t ore
    }
}

In [48]:
combined_explosives_df, inferred_explosives_df = engine.infer_explosives_for_sites(
    site_ids=site_id_to_fill_explosives,
    explosive_params=explosives_params,
)

## Land

In [51]:
site_id_to_fill_land = production_df[production_df['infer_land_data'] == 'Yes']['site_id'].tolist()

In [52]:
combined_land_df, inferred_land_df = engine.infer_land_for_sites(
    site_ids=site_id_to_fill_land,
    formula_open_pit="0.791 * ore_processed_t - 7.76e5",
    formula_underground="Uniform(5e5, 2e6)",
    formula_other="Uniform(1e4, 1e5)",
    overwrite=False
)


In [53]:
# Add NPV to land df, and put 'Unspecified NPV' for missing values
combined_land_df = combined_land_df.merge(production_df[['site_id', 'npv']], on='site_id', how='left')

In [54]:
combined_land_df['npv'] = combined_land_df['npv'].fillna('Unspecified NPV')

In [55]:
combined_land_df

,site_id,activity_name,mining_processing_type,archetypes,value,operation_periods,unit,data_source,value_formula,flow_type,subflow_type,parameter_distribution,npv
0,BC-MAIN-23155c25,NaN,Underground,NaN,1.499690e+06,1966–1985; 2002–2015; 2019–open,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,Unspecified NPV
1,BC-MAIN-3ef4f421,NaN,NaN,NaN,1.396089e+06,NaN,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,Unspecified NPV
2,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator",Cu Porphyry,7.967835e+06,NaN,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,Cold Evergreen Needleleaf Forest
3,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Au–Ag free-milling,4.167369e+05,NaN,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,Cold Evergreen Needleleaf Forest
4,BC-MAIN-599152a0,Open-pit mining and beneficiation at Copper Mo...,"Open-pit, concentrator",Cu Porphyry,1.323321e+07,1884–1958; 2011–open,m2,MetalliCan,MetalliCan,NaN,NaN,NaN,Cool Evergreen Needleleaf Forest
...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,ON-MAIN-f4fc3276,Underground mining and beneficiation at Sugar ...,"Underground, concentrator",Au–Ag free-milling,2.933388e+11,NaN,m2,Inference | land | underground,"ore_processed_t * Uniform(5e5, 2e6)",Land,Land area,"Uniform(5e5, 2e6)",Cool Mixed Forest
143,AB-MAIN-d3a4aba9,Refining at The Cobalt Refinery Company Inc.,Refinery,Co hydrometallurgical refinery,NaN,NaN,m2,Inference | land | other,"ore_processed_t * Uniform(1e4, 1e5)",Land,Land area,"Uniform(1e4, 1e5)",Cold Evergreen Needleleaf Forest
144,MB-MAIN-e0a6250e,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Ni–Cu sulfide,-4.084176e+05,NaN,m2,Inference | land | open-pit regression,0.791 * ore_processed_t - 7.76e5,Land,Land area,None,Cold Evergreen Needleleaf Forest
145,BC-MAIN-3bb6b7cd,Refining at Trail,Refinery,Zn refinery,NaN,NaN,m2,Inference | land | other,"ore_processed_t * Uniform(1e4, 1e5)",Land,Land area,"Uniform(1e4, 1e5)",Cool Evergreen Needleleaf Forest


# Integrate carbon stock change and water in the relevant dfs

## Multiply land area with carbon stock change and integrate in land_df

In [56]:
carbon_stock_df

,carbon_stock_ecosystems_id,pool,variable,unit,value,main_id,source_id,facility_group_id,site_id,activity_name,mining_processing_type,archetypes,data_source,value_formula
0,carbon_stock-35d0dc71-1,agbc,area_ha,ha,49.315624,AB-MAIN-35d0dc71,https://zenodo.org/records/15777016,<NA>,AB-MAIN-35d0dc71,NaN,NaN,NaN,MetalliCan,MetalliCan
1,carbon_stock-35d0dc71-2,agbc,mean_act,tC/ha,10.000000,AB-MAIN-35d0dc71,https://zenodo.org/records/15777016,<NA>,AB-MAIN-35d0dc71,NaN,NaN,NaN,MetalliCan,MetalliCan
2,carbon_stock-35d0dc71-3,agbc,mean_prim,tC/ha,36.000000,AB-MAIN-35d0dc71,https://zenodo.org/records/15777016,<NA>,AB-MAIN-35d0dc71,NaN,NaN,NaN,MetalliCan,MetalliCan
3,carbon_stock-35d0dc71-4,bgbc,area_ha,ha,49.315624,AB-MAIN-35d0dc71,https://zenodo.org/records/15777016,<NA>,AB-MAIN-35d0dc71,NaN,NaN,NaN,MetalliCan,MetalliCan
4,carbon_stock-35d0dc71-5,bgbc,mean_act,tC/ha,3.000000,AB-MAIN-35d0dc71,https://zenodo.org/records/15777016,<NA>,AB-MAIN-35d0dc71,NaN,NaN,NaN,MetalliCan,MetalliCan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3505,carbon_stock-e05ed9fe-9,soc,mean_prim,tC/ha,80.000000,YT-MAIN-e05ed9fe,https://zenodo.org/records/15777016,<NA>,YT-MAIN-e05ed9fe,NaN,NaN,NaN,MetalliCan,MetalliCan
3506,NaN,agbc,mean_variation,tC/ha,10.000000,YT-MAIN-e05ed9fe,https://zenodo.org/records/15777016,<NA>,YT-MAIN-e05ed9fe,NaN,NaN,NaN,MetalliCan,MetalliCan
3507,NaN,bgbc,mean_variation,tC/ha,4.000000,YT-MAIN-e05ed9fe,https://zenodo.org/records/15777016,<NA>,YT-MAIN-e05ed9fe,NaN,NaN,NaN,MetalliCan,MetalliCan
3508,NaN,soc,mean_variation,tC/ha,16.000000,YT-MAIN-e05ed9fe,https://zenodo.org/records/15777016,<NA>,YT-MAIN-e05ed9fe,NaN,NaN,NaN,MetalliCan,MetalliCan


In [57]:
# We integrate the carbon_stock_df in the biosphere_df
carbon_stock_df = carbon_stock_df[carbon_stock_df['pool'] == 'all']
carbon_stock_df.rename(columns={'value': 'carbon_variation_tC_ha'}, inplace=True)
carbon_stock_df['carbon_variation_CO2_m2'] = carbon_stock_df['carbon_variation_tC_ha'] * 44 / 12 / 10000  # convert from C to CO2
carbon_stock_df.drop(columns=['carbon_variation_tC_ha', 'unit', 'carbon_stock_ecosystems_id', 'pool',  ], inplace=True)
#carbon_stock_df.rename(columns={'carbon_variation_CO2_m2': 'value'}, inplace=True)
carbon_stock_df['flow_direction'] = 'Emission'
carbon_stock_df['compartment_name'] = 'Air, soil'
carbon_stock_df['release_pathway'] = ''
carbon_stock_df['unit'] = 't/m2'
carbon_stock_df['substance_name'] = "Carbon stock change"

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_3832\1528102575.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  carbon_stock_df.rename(columns={'value': 'carbon_variation_tC_ha'}, inplace=True)
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_3832\1528102575.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  carbon_stock_df['carbon_variation_CO2_m2'] = carbon_stock_df['carbon_variation_tC_ha'] * 44 / 12 / 10000  # convert from C to CO2
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_3832\1528102575.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a

In [58]:
# Merge with land_df to get the surface area
carbon_stock_df = carbon_stock_df.merge(combined_land_df[['site_id', 'value']], on='site_id', how='left')
#carbon_stock_df.rename(columns={'value': 'area_m2'}, inplace=True)

In [59]:
carbon_stock_df.rename(columns={'value': 'area_m2'}, inplace=True)

In [60]:
carbon_stock_df['value'] = carbon_stock_df['area_m2'] * carbon_stock_df['carbon_variation_CO2_m2']

In [61]:
carbon_stock_df = carbon_stock_df.dropna(subset=['value'])

In [62]:
carbon_stock_df

,variable,main_id,source_id,facility_group_id,site_id,activity_name,mining_processing_type,archetypes,data_source,value_formula,carbon_variation_CO2_m2,flow_direction,compartment_name,release_pathway,unit,substance_name,area_m2,value
12,mean_variation,BC-MAIN-23155c25,https://zenodo.org/records/15777016,<NA>,BC-MAIN-23155c25,NaN,NaN,NaN,MetalliCan,MetalliCan,0.024933,Emission,"Air, soil",,t/m2,Carbon stock change,1.499690e+06,37392.267311
17,mean_variation,BC-MAIN-3ef4f421,https://zenodo.org/records/15777016,<NA>,BC-MAIN-3ef4f421,NaN,NaN,NaN,MetalliCan,MetalliCan,0.017967,Emission,"Air, soil",,t/m2,Carbon stock change,1.396089e+06,25083.063522
18,mean_variation,BC-MAIN-3f490561,https://zenodo.org/records/15777016,<NA>,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator",Cu Porphyry,MetalliCan,MetalliCan,0.016500,Emission,"Air, soil",,t/m2,Carbon stock change,7.967835e+06,131469.271520
19,mean_variation,BC-MAIN-4724f4ba,https://zenodo.org/records/15777016,<NA>,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Au–Ag free-milling,MetalliCan,MetalliCan,0.019433,Emission,"Air, soil",,t/m2,Carbon stock change,4.167369e+05,8098.586319
22,mean_variation,BC-MAIN-599152a0,https://zenodo.org/records/15777016,<NA>,BC-MAIN-599152a0,Open-pit mining and beneficiation at Copper Mo...,"Open-pit, concentrator",Cu Porphyry,MetalliCan,MetalliCan,0.018700,Emission,"Air, soil",,t/m2,Carbon stock change,1.323321e+07,247461.034267
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258,mean_variation,SK-MAIN-9dd2b7f8,https://zenodo.org/records/15777016,<NA>,SK-MAIN-9dd2b7f8,NaN,NaN,NaN,MetalliCan,MetalliCan,0.000000,Emission,"Air, soil",,t/m2,Carbon stock change,4.345047e+06,0.000000
259,mean_variation,SK-MAIN-bb89158f,https://zenodo.org/records/15777016,<NA>,SK-MAIN-bb89158f,NaN,NaN,NaN,MetalliCan,MetalliCan,0.009167,Emission,"Air, soil",,t/m2,Carbon stock change,1.023565e+07,93826.765900
260,mean_variation,SK-MAIN-d3c471e8,https://zenodo.org/records/15777016,<NA>,SK-MAIN-d3c471e8,NaN,NaN,NaN,MetalliCan,MetalliCan,0.008800,Emission,"Air, soil",,t/m2,Carbon stock change,1.973892e+06,17370.253608
267,mean_variation,YT-MAIN-44857446,https://zenodo.org/records/15777016,<NA>,YT-MAIN-44857446,Underground mining and beneficiation at Keno H...,"Underground, concentrator",High-grade polymetallic Pb-Zn-Ag,MetalliCan,MetalliCan,0.005133,Emission,"Air, soil",,t/m2,Carbon stock change,5.293594e+06,27173.782646


## Create water df and integrate in biosphere df

In [63]:
water_df = combined_material_df[combined_material_df['flow_type'] == 'Water']

In [64]:
water_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula,parameter_distribution
78,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Au–Ag free-milling,Water,Water,1.035403e+05,m3,Archetype inference | nan,ore_processed_t * 0.6237369273366383,NaN
84,QC-MAIN-e7e6a960,Open-pit mining and beneficiation at Canadian ...,"Open-pit, concentrator",Au–Ag free-milling,Water,Water,1.081117e+07,m3,Archetype inference | nan,ore_processed_t * 0.6237369273366383,NaN
89,ON-MAIN-f080c409,Underground mining at Copper Cliff Complex (mine),Underground,Ni–Cu sulfide,Water,Water,NaN,m3,Archetype inference | nan,ore_processed_t * nan,NaN
94,BC-MAIN-599152a0,Open-pit mining and beneficiation at Copper Mo...,"Open-pit, concentrator",Cu Porphyry,Water,Water,6.336982e+06,m3,Archetype inference | nan,ore_processed_t * 0.9234686820974333,NaN
100,ON-MAIN-52224e1e,Underground mining at Creighton,Underground,Ni–Cu sulfide,Water,Water,NaN,m3,Archetype inference | nan,ore_processed_t * nan,NaN
107,ON-MAIN-aeafbb59,Open-pit mining and beneficiation at Detour Lake,"Open-pit, concentrator",Au–Ag free-milling,Water,Water,1.586466e+07,m3,Archetype inference | nan,ore_processed_t * 0.6237369273366383,NaN
113,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Au–Ag free-milling,Water,Water,2.073613e+04,m3,Archetype inference | nan,ore_processed_t * 0.6237369273366383,NaN
119,ON-MAIN-4e0734b5,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Au–Ag free-milling,Water,Water,2.850478e+05,m3,Archetype inference | nan,ore_processed_t * 0.6237369273366383,NaN
124,ON-MAIN-206041d1,Underground mining at Fraser,Underground,Ni–Cu sulfide,Water,Water,NaN,m3,Archetype inference | nan,ore_processed_t * nan,NaN
130,ON-MAIN-48fe2205,Underground mining at Garson,Underground,Ni–Cu sulfide,Water,Water,NaN,m3,Archetype inference | nan,ore_processed_t * nan,NaN


In [65]:
water_df['substance_name'] = 'Water'
water_df['compartment_name'] = 'Water'
water_df['flow_direction'] = 'Consumption'
water_df['release_pathway'] = ''

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_3832\1603994030.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  water_df['substance_name'] = 'Water'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_3832\1603994030.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  water_df['compartment_name'] = 'Water'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_3832\1603994030.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea

In [66]:
water_df = water_df[biosphere_col]

In [67]:
biosphere_df = pd.concat([biosphere_df, water_df])

# Normalize flows

In [68]:
combined_explosives_df = combined_explosives_df[combined_explosives_df['flow_type'] != 'Water']

In [69]:
combined_explosives_df['flow_type'].unique()

array(['Material use', 'Materials', 'Material'], dtype=object)

In [70]:
combined_explosives_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula,parameter_distribution
0,QC-MAIN-b86f7d07,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Au–Ag free-milling,Material use,Surface/underground emulsion & ANFO,2968.123704,t,MetalliCan,MetalliCan,NaN
1,ON-MAIN-aeafbb59,Open-pit mining and beneficiation at Detour Lake,"Open-pit, concentrator",Au–Ag free-milling,Material use,Explosives,16268.500000,t,MetalliCan,MetalliCan,NaN
2,ON-MAIN-cb85213a,Underground mining and beneficiation at Eagle ...,"Underground, concentrator",Au–Ag free-milling,Material use,Explosives,1211.000000,t,MetalliCan,MetalliCan,NaN
3,QC-MAIN-6dc537e6,Underground mining and beneficiation at Éléonore,"Underground, concentrator",Au–Ag free-milling,Material use,Cement,27374.000000,t,MetalliCan,MetalliCan,NaN
4,QC-MAIN-6dc537e6,Underground mining and beneficiation at Éléonore,"Underground, concentrator",Au–Ag free-milling,Material use,Grinding media,3039.900000,t,MetalliCan,MetalliCan,NaN
...,...,...,...,...,...,...,...,...,...,...,...
381,GRP-91aaa60b,Underground mining and beneficiation at Cigar ...,"Underground, concentrator",High-grade U mine + mill,Material,Explosives,8745.371238,kg,Inference | explosives | underground,"ore_processed_t * Uniform(0.1, 0.3)","Uniform(0.1, 0.3)"
382,GRP-21eee27d,Underground mining and beneficiation at Key La...,"Underground, concentrator",High-grade U mine + mill,Material,Explosives,19681.774207,kg,Inference | explosives | underground,"ore_processed_t * Uniform(0.1, 0.3)","Uniform(0.1, 0.3)"
383,GRP-147b3123,Underground mining and beneficiation at Timmin...,"Underground, concentrator",Au–Ag free-milling,Material,Explosives,314800.000000,kg,Inference | explosives | underground,"ore_processed_t * Uniform(0.1, 0.3)","Uniform(0.1, 0.3)"
384,GRP-14bfbb82,Underground mining and beneficiation at Seabee...,"Underground, concentrator",Au–Ag free-milling,Material,Explosives,24400.000000,kg,Inference | explosives | underground,"ore_processed_t * Uniform(0.1, 0.3)","Uniform(0.1, 0.3)"


In [71]:
combined_material_df = pd.concat([combined_explosives_df, tailings_df], ignore_index=True)

In [72]:
combined_material_df['flow_type'].unique()

array(['Material use', 'Materials', 'Material'], dtype=object)

In [73]:
from core.normalization_allocation import normalize_flows, normalize_land_flows

### Per ore processed

In [74]:
energy_ore = normalize_flows(energy_df, production_df, mode='ore', value_col='value')
material_ore = normalize_flows(material_df, production_df, mode='ore', value_col='value')
biosphere_ore = normalize_flows(biosphere_df, production_df, mode='ore', value_col='value')

### Per concentrate stream

In [75]:
energy_conc_econ = normalize_flows(combined_energy_df, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value')

In [76]:
material_conc_econ = normalize_flows(combined_material_df, production_df, price_df=price_df,  mode='concentrate', allocation='economic', value_col='value')

In [77]:
biosphere_conc_econ = normalize_flows(biosphere_df, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value')

In [78]:
land_conc_econ = normalize_land_flows(combined_land_df, production_df, price_df=price_df, mode='concentrate', allocation='economic')

In [79]:
carbon_stock_conc_econ = normalize_land_flows(carbon_stock_df, production_df, price_df=price_df, mode='concentrate', allocation='economic')

In [80]:
land_conc_econ.loc[
    land_conc_econ["functional_unit"].str.lower().eq("au concentrate"),
    "functional_unit"
] = "Doré"

In [81]:
carbon_stock_conc_econ.loc[
    carbon_stock_conc_econ["functional_unit"].str.lower().eq("au concentrate"),
    "functional_unit"
] = "Doré"

In [82]:
carbon_stock_conc_econ = carbon_stock_conc_econ[carbon_stock_conc_econ['flow_type'] == 'transformation_from']
carbon_stock_conc_econ['flow_type'] = 'Carbon stock change'
carbon_stock_conc_econ['unit'] = 't'

# Exports normalized dataframes

In [83]:
energy_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/energy_df.csv', index=False)
material_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/material_df.csv', index=False)
biosphere_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/biosphere_df.csv', index=False)

In [84]:
energy_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/energy_df.csv', index=False)
material_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/material_df.csv', index=False)
biosphere_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/biosphere_df.csv', index=False)
land_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/land_df.csv', index=False)
carbon_stock_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/carbon_stock_df.csv', index=False)